# Nonlinear Model Comparison

This notebook compares the selected Ridge baseline with Random Forest models using the prepared player-season data, chronological splits, and selected features from the Ridge notebook.

Most reusable logic now lives in `src/prem_valuation/`, so this notebook focuses on the experiment flow and the resulting 2025/26 ranking outputs.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
src_path = project_root / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import pandas as pd

from prem_valuation.data import (
    PROCESSED_DIR,
    load_appearances,
    load_games,
    load_interim_tables,
    load_players,
)
from prem_valuation.features import (
    HISTORY_FEATURES,
    HISTORY_NUMERIC_FEATURES,
    SELECTED_CATEGORICAL_FEATURES,
    SELECTED_FEATURES,
    SELECTED_NUMERIC_FEATURES,
    TARGET,
    add_engineered_features,
    add_latest_history_to_scoring,
    add_previous_season_features,
)
from prem_valuation.modeling import (
    evaluate_predictions,
    make_random_forest_model,
    make_ridge_model,
    predict_non_negative,
    walk_forward_evaluate,
)
from prem_valuation.rankings import (
    attach_current_player_values,
    attach_season_club_metadata,
    build_season_club_lookup,
    build_scoring_results,
    make_ranking_tables,
    save_ranking_outputs,
)

## Load prepared data

In [ ]:
model_data, scoring_data = load_interim_tables()

model_data = add_engineered_features(model_data)
scoring_data = add_engineered_features(scoring_data)

model_data.shape, scoring_data.shape

## Chronological split

In [ ]:
train_data = model_data.loc[model_data["season"].between(2016, 2022)].copy()
validation_data = model_data.loc[model_data["season"].eq(2023)].copy()
test_data = model_data.loc[model_data["season"].eq(2024)].copy()

for name, dataset in {
    "train": train_data,
    "validation": validation_data,
    "test": test_data,
    "scoring": scoring_data,
}.items():
    print(name, dataset.shape, sorted(dataset["season"].unique()))

## Model builders

In [ ]:
ridge_builder = lambda: make_ridge_model(
    SELECTED_NUMERIC_FEATURES,
    SELECTED_CATEGORICAL_FEATURES,
    alpha=100,
)

forest_builder = lambda: make_random_forest_model(
    SELECTED_NUMERIC_FEATURES,
    SELECTED_CATEGORICAL_FEATURES,
)

history_forest_builder = lambda: make_random_forest_model(
    HISTORY_NUMERIC_FEATURES,
    SELECTED_CATEGORICAL_FEATURES,
)

## Baseline nonlinear comparison

Compare the frozen Ridge baseline against a plain Random Forest using walk-forward validation.

In [ ]:
ridge_cv = walk_forward_evaluate(
    model_data,
    SELECTED_FEATURES,
    TARGET,
    ridge_builder,
)
forest_cv = walk_forward_evaluate(
    model_data,
    SELECTED_FEATURES,
    TARGET,
    forest_builder,
)

model_comparison = pd.DataFrame({
    "ridge": ridge_cv[["mae", "rmse", "r2"]].mean(),
    "random_forest": forest_cv[["mae", "rmse", "r2"]].mean(),
}).T

model_comparison

In [ ]:
fold_comparison = ridge_cv.merge(
    forest_cv,
    on="validation_season",
    suffixes=("_ridge", "_forest"),
)
fold_comparison["mae_improvement"] = (
    fold_comparison["mae_ridge"] - fold_comparison["mae_forest"]
)

fold_comparison

## Final Random Forest test evaluation

Random Forest was selected using walk-forward validation only. We now evaluate it once on the untouched 2024/25 test season.

In [ ]:
development_data = pd.concat(
    [train_data, validation_data],
    ignore_index=True,
)

final_forest_model = forest_builder()
final_forest_model.fit(
    development_data[SELECTED_FEATURES],
    development_data[TARGET],
)

test_predictions = predict_non_negative(
    final_forest_model,
    test_data[SELECTED_FEATURES],
)

pd.Series(
    evaluate_predictions(test_data[TARGET], test_predictions),
    name="2024/25 final test",
)

## Random Forest residual analysis

Inspect the model's biggest misses to see whether the nonlinear model reduces the Ridge baseline's tendency to compress elite players toward the average.

In [ ]:
test_results = test_data[
    [
        "season",
        "player_id",
        "player_name",
        "position",
        "sub_position",
        "age_at_season_end",
        "minutes_played",
        TARGET,
    ]
].copy()

test_results["predicted_value"] = test_predictions
test_results["residual"] = test_results[TARGET] - test_results["predicted_value"]
test_results["absolute_error"] = test_results["residual"].abs()

test_results.sort_values("absolute_error", ascending=False).head(15)

## Add historical features

The first Random Forest only sees the current season. We now add previous-season performance and previous market value so the model has memory of player reputation and recent history.

In [ ]:
model_data_with_history = add_previous_season_features(model_data)

model_data_with_history[
    [
        "player_name",
        "season",
        "minutes_played",
        "previous_minutes_played",
        TARGET,
        "previous_market_value_in_eur",
    ]
].head(10)

## History model comparison

Compare the plain Random Forest against the history-aware Random Forest using the same walk-forward folds.

In [ ]:
history_forest_cv = walk_forward_evaluate(
    model_data_with_history,
    HISTORY_FEATURES,
    TARGET,
    history_forest_builder,
)

pd.DataFrame({
    "ridge": ridge_cv[["mae", "rmse", "r2"]].mean(),
    "random_forest": forest_cv[["mae", "rmse", "r2"]].mean(),
    "history_random_forest": history_forest_cv[["mae", "rmse", "r2"]].mean(),
}).T

In [ ]:
history_fold_comparison = forest_cv.merge(
    history_forest_cv,
    on="validation_season",
    suffixes=("_forest", "_history_forest"),
)
history_fold_comparison["mae_improvement"] = (
    history_fold_comparison["mae_forest"]
    - history_fold_comparison["mae_history_forest"]
)

history_fold_comparison

## Final history Random Forest test evaluation

The history Random Forest was selected using walk-forward validation. We now evaluate it once on the untouched 2024/25 test season.

In [ ]:
development_data_with_history = model_data_with_history.loc[
    model_data_with_history["season"].between(2016, 2023)
].copy()
test_data_with_history = model_data_with_history.loc[
    model_data_with_history["season"].eq(2024)
].copy()

final_history_forest_model = history_forest_builder()
final_history_forest_model.fit(
    development_data_with_history[HISTORY_FEATURES],
    development_data_with_history[TARGET],
)

history_test_predictions = predict_non_negative(
    final_history_forest_model,
    test_data_with_history[HISTORY_FEATURES],
)

pd.Series(
    evaluate_predictions(test_data_with_history[TARGET], history_test_predictions),
    name="2024/25 final test",
)

In [ ]:
history_test_results = test_data_with_history[
    [
        "season",
        "player_id",
        "player_name",
        "position",
        "sub_position",
        "age_at_season_end",
        "minutes_played",
        "previous_minutes_played",
        TARGET,
        "previous_market_value_in_eur",
    ]
].copy()

history_test_results["predicted_value"] = history_test_predictions
history_test_results["residual"] = (
    history_test_results[TARGET] - history_test_results["predicted_value"]
)
history_test_results["absolute_error"] = history_test_results["residual"].abs()

history_test_results.sort_values("absolute_error", ascending=False).head(15)

In [ ]:
history_test_results["has_previous_market_value"] = (
    history_test_results["previous_market_value_in_eur"].notna()
)

history_test_results.groupby("has_previous_market_value").agg(
    players=("player_id", "count"),
    actual_mean=(TARGET, "mean"),
    predicted_mean=("predicted_value", "mean"),
    mae=("absolute_error", "mean"),
    mean_residual=("residual", "mean"),
)

## Score 2025/26 players

Apply the selected history Random Forest model to the 2025/26 scoring population. The prediction is an expected Transfermarkt-style value; the valuation gap is the model prediction minus the current Transfermarkt value.

In [ ]:
players = load_players()
season_club_lookup = build_season_club_lookup(
    load_appearances(),
    load_games(),
    players,
    season=2025,
)

scoring_data_with_history = add_latest_history_to_scoring(
    scoring_data,
    model_data,
    history_season=2024,
)
scoring_data_with_history = attach_season_club_metadata(
    scoring_data_with_history,
    season_club_lookup,
)
scoring_data_with_history = attach_current_player_values(
    scoring_data_with_history,
    players,
)

scoring_data_with_history[
    [
        "player_name",
        "season_club_name",
        "current_club_name",
        "minutes_played",
        "previous_market_value_in_eur",
        "current_market_value_in_eur",
    ]
].head(10)

In [ ]:
scoring_predictions = predict_non_negative(
    final_history_forest_model,
    scoring_data_with_history[HISTORY_FEATURES],
)
scoring_results = build_scoring_results(
    scoring_data_with_history,
    scoring_predictions,
)

scoring_results[
    [
        "player_name",
        "season_club_name",
        "current_club_name",
        "position",
        "minutes_played",
        "current_market_value_in_eur",
        "predicted_value",
        "valuation_gap",
        "has_previous_market_value",
        "minutes_band",
    ]
].head()

## 2025/26 valuation-gap rankings

Filter to current Premier League players with current Transfermarkt values, then rank regular-minute players by valuation gap.

In [ ]:
ranking_outputs = make_ranking_tables(scoring_results)

undervalued_players = ranking_outputs["undervalued_players_2025_26"]
overvalued_players = ranking_outputs["overvalued_players_2025_26"]
high_confidence_undervalued = ranking_outputs[
    "high_confidence_undervalued_2025_26"
]
recruitment_bargains = ranking_outputs["recruitment_bargains_2025_26"]

undervalued_players

In [ ]:
overvalued_players

## High-confidence bargain views

Focus on regular-minute players with previous market values. This removes many new-arrival cases where the model has less historical context.

In [ ]:
high_confidence_undervalued

In [ ]:
recruitment_bargains

## Save ranking outputs

Save the generated 2025/26 scoring and ranking tables to `data/processed/` so they can be used by later notebooks, reports, or app views without rerunning the analysis manually.

In [ ]:
save_ranking_outputs(
    ranking_outputs,
    PROCESSED_DIR,
)

## Experiment log

- Random Forest beat the selected Ridge baseline on walk-forward validation and final 2024/25 test performance.
- Adding previous-season performance and previous market value produced the largest improvement, reducing final test MAE from about EUR 9.46m to EUR 5.80m.
- The history model is strongest for players with previous Premier League market values and weaker for new arrivals or breakout players without prior PL history.
- 2025/26 scoring produced three useful ranking views: candidate undervalued players, candidate overvalued players, and a recruitment-style bargain shortlist.